In [34]:
#!/usr/bin/env python3

import warnings
import os
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================
DATASETS = [
    ("testset_A_Ak_CAL_embeddings_95.npy", "testset_A_Ak_CAL_table_95.xlsx"),
    ("testset_A_Ak_CAL_embeddings_90.npy", "testset_A_Ak_CAL_table_90.xlsx"),
    ("testset_A_Ak_CAL_embeddings_85.npy", "testset_A_Ak_CAL_table_85.xlsx"),
]

RANDOM_STATE = 42
N_SPLITS = 5

OUTPUT_DIR = "Ak_CAL_binary_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# 1. LABEL NORMALIZATION
# =========================================================
def normalize_label(x):
    s = str(x).strip().upper().replace("-", "_").replace(" ", "_")

    if s in {"A", "A_NORMAL", "ANORMAL", "A_NORMAL_DOMAIN"}:
        return "A_normal"
    if s in {"AK", "AK_DOMAIN", "AMP", "AMP_BINDING", "AMPBINDING"}:
        return "Ak"
    if s in {"CAL", "CAL_DOMAIN", "CALDOMAIN"}:
        return "CAL"

    return str(x).strip()


# =========================================================
# 2. LOAD METADATA
# =========================================================
def load_metadata(metadata_file):
    if metadata_file.endswith(".xlsx"):
        df = pd.read_excel(metadata_file)
    elif metadata_file.endswith(".csv"):
        df = pd.read_csv(metadata_file)
    elif metadata_file.endswith(".tsv"):
        df = pd.read_csv(metadata_file, sep="\t")
    else:
        raise ValueError(f"Unsupported metadata file format: {metadata_file}")

    if "domain_label" in df.columns:
        df["domain_label"] = df["domain_label"].astype(str).str.strip()
    elif "domain_type" in df.columns:
        df["domain_label"] = df["domain_type"].astype(str).str.strip()
    else:
        raise ValueError(
            f"Could not find 'domain_label' or 'domain_type' column in {metadata_file}"
        )

    df["domain_label"] = df["domain_label"].apply(normalize_label)

    if "id" not in df.columns:
        df["id"] = [f"seq_{i}" for i in range(len(df))]

    return df


# =========================================================
# 3. DEFINE MODEL BUILDERS
# =========================================================
def make_model(model_name, weighted=False):
    if model_name == "LogisticRegression":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=5000,
                class_weight="balanced" if weighted else None,
                random_state=RANDOM_STATE
            ))
        ])

    elif model_name == "SVM":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                probability=True,
                class_weight="balanced" if weighted else None,
                random_state=RANDOM_STATE
            ))
        ])

    elif model_name == "RandomForest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced" if weighted else None,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])

    elif model_name == "XGBoost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])

    else:
        raise ValueError(f"Unknown model: {model_name}")


model_names = ["LogisticRegression", "SVM", "RandomForest", "XGBoost"]


# =========================================================
# 4. MAIN LOOP
# =========================================================
all_overall_results = []
all_per_class_results = []

for EMBEDDING_FILE, METADATA_FILE in DATASETS:
    print("\n" + "#" * 80)
    print("Processing dataset")
    print("Embedding file :", EMBEDDING_FILE)
    print("Metadata file  :", METADATA_FILE)
    print("#" * 80)

    # -----------------------------
    # Load embeddings
    # -----------------------------
    X = np.load(EMBEDDING_FILE)
    print("Loaded embedding shape:", X.shape)

    # -----------------------------
    # Load metadata
    # -----------------------------
    df = load_metadata(METADATA_FILE)
    print("Loaded metadata rows:", len(df))
    print("Metadata columns:", df.columns.tolist())

    if len(df) != X.shape[0]:
        raise ValueError(
            f"Metadata rows ({len(df)}) do not match embedding rows ({X.shape[0]}) "
            f"for dataset: {EMBEDDING_FILE}"
        )

    # -----------------------------
    # Keep only Ak and CAL
    # -----------------------------
    target_classes = ["Ak", "CAL"]
    keep_idx = df[df["domain_label"].isin(target_classes)].index

    df = df.loc[keep_idx].copy().reset_index(drop=True)
    X_bin = X[keep_idx]

    print("\nBinary label counts:")
    print(df["domain_label"].value_counts())

    if len(df) != X_bin.shape[0]:
        raise ValueError("Filtered metadata and embedding sizes do not match.")

    unique_classes = sorted(df["domain_label"].unique())
    if len(unique_classes) != 2:
        raise ValueError(
            f"Expected 2 classes (Ak and CAL), but found: {unique_classes}"
        )

    # -----------------------------
    # Encode labels
    # -----------------------------
    le = LabelEncoder()
    y = le.fit_transform(df["domain_label"])

    print("\nEncoded classes:")
    for i, cls in enumerate(le.classes_):
        print(f"{i} -> {cls}")

    # make sure positive class is Ak if possible
    if list(le.classes_) != ["Ak", "CAL"]:
        print("\nWarning: class order is not ['Ak', 'CAL']. Current order:", list(le.classes_))

    # -----------------------------
    # Output file names
    # -----------------------------
    base_name = os.path.splitext(os.path.basename(EMBEDDING_FILE))[0]

    OUTPUT_OVERALL_XLSX = os.path.join(OUTPUT_DIR, f"{base_name}_Ak_CAL_overall_results.xlsx")
    OUTPUT_OVERALL_TSV = os.path.join(OUTPUT_DIR, f"{base_name}_Ak_CAL_overall_results.tsv")
    OUTPUT_PER_CLASS_XLSX = os.path.join(OUTPUT_DIR, f"{base_name}_Ak_CAL_per_class_results.xlsx")
    OUTPUT_PREDICTIONS_XLSX = os.path.join(OUTPUT_DIR, f"{base_name}_Ak_CAL_fold_predictions.xlsx")
    OUTPUT_LABEL_MAP_XLSX = os.path.join(OUTPUT_DIR, f"{base_name}_Ak_CAL_label_mapping.xlsx")

    # -----------------------------
    # Cross-validation
    # -----------------------------
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    overall_results = []
    per_class_results = []
    all_fold_predictions = []

    for weighting in [False, True]:
        weight_mode = "weighted" if weighting else "unweighted"

        print("\n" + "=" * 60)
        print("Running:", weight_mode)
        print("=" * 60)

        for model_name in model_names:
            print(f"\nModel: {model_name}")

            accs, f1s, precs, recs, aucs = [], [], [], [], []
            all_y_true = []
            all_y_pred = []

            for fold, (train_idx, test_idx) in enumerate(cv.split(X_bin, y), start=1):
                X_train, X_test = X_bin[train_idx], X_bin[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]
                test_ids = df.loc[test_idx, "id"].tolist()

                model = make_model(model_name=model_name, weighted=weighting)

                if model_name == "XGBoost" and weighting:
                    sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
                    model.fit(X_train, y_train, clf__sample_weight=sample_weights)
                else:
                    model.fit(X_train, y_train)

                y_pred = model.predict(X_test)

                if hasattr(model, "predict_proba"):
                    y_score = model.predict_proba(X_test)[:, 1]
                else:
                    y_score = model.decision_function(X_test)

                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred, average="binary", zero_division=0)
                prec = precision_score(y_test, y_pred, average="binary", zero_division=0)
                rec = recall_score(y_test, y_pred, average="binary", zero_division=0)
                auc = roc_auc_score(y_test, y_score)

                accs.append(acc)
                f1s.append(f1)
                precs.append(prec)
                recs.append(rec)
                aucs.append(auc)

                all_y_true.extend(y_test.tolist())
                all_y_pred.extend(y_pred.tolist())

                fold_pred_df = pd.DataFrame({
                    "Embedding_file": base_name,
                    "Metadata_file": os.path.basename(METADATA_FILE),
                    "Weight_mode": weight_mode,
                    "Model": model_name,
                    "fold": fold,
                    "id": test_ids,
                    "true_label": le.inverse_transform(y_test),
                    "pred_label": le.inverse_transform(y_pred)
                })
                all_fold_predictions.append(fold_pred_df)

                print(
                    f"  Fold {fold}: "
                    f"ACC={acc:.4f}, "
                    f"F1={f1:.4f}, "
                    f"PREC={prec:.4f}, "
                    f"REC={rec:.4f}, "
                    f"ROC_AUC={auc:.4f}"
                )

            overall_results.append({
                "Embedding_file": base_name,
                "Metadata_file": os.path.basename(METADATA_FILE),
                "Weight_mode": weight_mode,
                "Model": model_name,
                "Accuracy_mean": np.mean(accs),
                "Accuracy_std": np.std(accs),
                "F1_mean": np.mean(f1s),
                "F1_std": np.std(f1s),
                "Precision_mean": np.mean(precs),
                "Precision_std": np.std(precs),
                "Recall_mean": np.mean(recs),
                "Recall_std": np.std(recs),
                "ROC_AUC_mean": np.mean(aucs),
                "ROC_AUC_std": np.std(aucs),
            })

            report = classification_report(
                all_y_true,
                all_y_pred,
                labels=list(range(len(le.classes_))),
                target_names=le.classes_,
                output_dict=True,
                zero_division=0
            )

            for cls_name in le.classes_:
                per_class_results.append({
                    "Embedding_file": base_name,
                    "Metadata_file": os.path.basename(METADATA_FILE),
                    "Weight_mode": weight_mode,
                    "Model": model_name,
                    "Class": cls_name,
                    "Precision": report[cls_name]["precision"],
                    "Recall": report[cls_name]["recall"],
                    "F1_score": report[cls_name]["f1-score"],
                    "Support": report[cls_name]["support"]
                })

            print("\nPer-class metrics:")
            for cls_name in le.classes_:
                print(
                    f"  {cls_name}: "
                    f"Precision={report[cls_name]['precision']:.4f}, "
                    f"Recall={report[cls_name]['recall']:.4f}, "
                    f"F1={report[cls_name]['f1-score']:.4f}, "
                    f"Support={report[cls_name]['support']:.0f}"
                )

    # -----------------------------
    # Save per-dataset results
    # -----------------------------
    overall_df = pd.DataFrame(overall_results)
    overall_df = overall_df.sort_values(
        by=["F1_mean", "ROC_AUC_mean", "Accuracy_mean"],
        ascending=False
    ).reset_index(drop=True)

    per_class_df = pd.DataFrame(per_class_results)
    pred_df = pd.concat(all_fold_predictions, ignore_index=True)

    label_map_df = pd.DataFrame({
        "encoded_label": list(range(len(le.classes_))),
        "domain_class": le.classes_
    })

    overall_df.to_excel(OUTPUT_OVERALL_XLSX, index=False)
    overall_df.to_csv(OUTPUT_OVERALL_TSV, sep="\t", index=False)
    per_class_df.to_excel(OUTPUT_PER_CLASS_XLSX, index=False)
    pred_df.to_excel(OUTPUT_PREDICTIONS_XLSX, index=False)
    label_map_df.to_excel(OUTPUT_LABEL_MAP_XLSX, index=False)

    print("\nSaved:")
    print(" -", OUTPUT_OVERALL_XLSX)
    print(" -", OUTPUT_OVERALL_TSV)
    print(" -", OUTPUT_PER_CLASS_XLSX)
    print(" -", OUTPUT_PREDICTIONS_XLSX)
    print(" -", OUTPUT_LABEL_MAP_XLSX)

    all_overall_results.append(overall_df)
    all_per_class_results.append(per_class_df)


# =========================================================
# 5. SAVE COMBINED RESULTS
# =========================================================
if len(all_overall_results) > 0:
    combined_overall_df = pd.concat(all_overall_results, ignore_index=True)
    combined_overall_df.to_excel(
        os.path.join(OUTPUT_DIR, "combined_Ak_CAL_overall_results.xlsx"),
        index=False
    )
    combined_overall_df.to_csv(
        os.path.join(OUTPUT_DIR, "combined_Ak_CAL_overall_results.tsv"),
        sep="\t",
        index=False
    )

if len(all_per_class_results) > 0:
    combined_per_class_df = pd.concat(all_per_class_results, ignore_index=True)
    combined_per_class_df.to_excel(
        os.path.join(OUTPUT_DIR, "combined_Ak_CAL_per_class_results.xlsx"),
        index=False
    )

print("\nAll done.")


################################################################################
Processing dataset
Embedding file : testset_A_Ak_CAL_embeddings_95.npy
Metadata file  : testset_A_Ak_CAL_table_95.xlsx
################################################################################
Loaded embedding shape: (518, 1280)
Loaded metadata rows: 518
Metadata columns: ['id', 'sequence', 'domain_label']

Binary label counts:
domain_label
CAL    215
Ak     119
Name: count, dtype: int64

Encoded classes:
0 -> Ak
1 -> CAL

Running: unweighted

Model: LogisticRegression
  Fold 1: ACC=0.8209, F1=0.8571, PREC=0.8780, REC=0.8372, ROC_AUC=0.8866
  Fold 2: ACC=0.7463, F1=0.8211, PREC=0.7500, REC=0.9070, ROC_AUC=0.7074
  Fold 3: ACC=0.7164, F1=0.7765, PREC=0.7857, REC=0.7674, ROC_AUC=0.7248
  Fold 4: ACC=0.7761, F1=0.8315, PREC=0.8043, REC=0.8605, ROC_AUC=0.8043
  Fold 5: ACC=0.7879, F1=0.8333, PREC=0.8537, REC=0.8140, ROC_AUC=0.7594

Per-class metrics:
  Ak: Precision=0.6875, Recall=0.6471, F1=0.6667, Su

In [35]:
#!/usr/bin/env python3

import warnings
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

from Bio import pairwise2

warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================
DATASETS = [
    "testset_A_Ak_CAL_table_95.xlsx",
    "testset_A_Ak_CAL_table_90.xlsx",
    "testset_A_Ak_CAL_table_85.xlsx",
]

OUTPUT_DIR = "Ak_CAL_sequence_identity_baseline_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5

TARGET_CLASSES = ["Ak", "CAL"]
LABEL_ORDER = ["Ak", "CAL"]


# =========================================================
# 1. LABEL NORMALIZATION
# =========================================================
def normalize_label(x):
    s = str(x).strip().upper().replace("-", "_").replace(" ", "_")

    if s in {"A", "A_NORMAL", "ANORMAL", "A_NORMAL_DOMAIN"}:
        return "A_normal"

    if s in {"AK", "AK_DOMAIN", "AMP", "AMP_BINDING", "AMPBINDING"}:
        return "Ak"

    if s in {"CAL", "CAL_DOMAIN", "CALDOMAIN"}:
        return "CAL"

    return str(x).strip()


# =========================================================
# 2. LOAD METADATA
# =========================================================
def load_metadata(metadata_file):
    if metadata_file.endswith(".xlsx"):
        df = pd.read_excel(metadata_file)
    elif metadata_file.endswith(".csv"):
        df = pd.read_csv(metadata_file)
    elif metadata_file.endswith(".tsv"):
        df = pd.read_csv(metadata_file, sep="\t")
    else:
        raise ValueError(f"Unsupported metadata file format: {metadata_file}")

    print("=" * 60)
    print("Loaded metadata file:", metadata_file)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print("=" * 60)

    required_cols = ["sequence"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {metadata_file}: {missing}")

    if "domain_label" in df.columns:
        label_col = "domain_label"
    elif "domain_type" in df.columns:
        label_col = "domain_type"
    else:
        raise ValueError(
            f"Could not find 'domain_label' or 'domain_type' column in {metadata_file}"
        )

    if "id" not in df.columns:
        df["id"] = [f"seq_{i}" for i in range(len(df))]

    df["domain_label"] = df[label_col].apply(normalize_label)

    df["sequence"] = (
        df["sequence"]
        .astype(str)
        .str.upper()
        .str.replace(r"[^A-Z]", "", regex=True)
    )

    return df


# =========================================================
# 3. SEQUENCE IDENTITY FUNCTION
# =========================================================
def calc_sequence_identity(seq1, seq2):
    """
    Global sequence identity baseline.
    Identity = matches / alignment length
    """
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True)[0]
    aligned_seq1 = aln.seqA
    aligned_seq2 = aln.seqB

    matches = sum(a == b for a, b in zip(aligned_seq1, aligned_seq2))
    aln_len = len(aligned_seq1)

    if aln_len == 0:
        return 0.0

    return matches / aln_len


# =========================================================
# 4. MAIN LOOP
# =========================================================
all_results = []
all_per_class_summary = []

for METADATA_FILE in DATASETS:
    print("\n" + "#" * 80)
    print("Processing:", METADATA_FILE)
    print("#" * 80)

    df = load_metadata(METADATA_FILE)

    # Keep only Ak and CAL
    df = df[df["domain_label"].isin(TARGET_CLASSES)].copy()
    df = df[df["sequence"] != ""].copy().reset_index(drop=True)

    print("\nBinary label counts:")
    print(df["domain_label"].value_counts())

    if df["domain_label"].nunique() != 2:
        raise ValueError(
            f"Expected 2 classes (Ak and CAL), found: {sorted(df['domain_label'].unique())}"
        )

    # Optional deduplication
    before = len(df)
    df = df.drop_duplicates(subset=["sequence", "domain_label"]).reset_index(drop=True)
    after = len(df)

    print(f"\nRemoved exact duplicate sequence+label rows: {before - after}")
    print("Rows after deduplication:", len(df))

    # Encode labels: Ak=0, CAL=1
    label_to_int = {label: i for i, label in enumerate(LABEL_ORDER)}
    int_to_label = {i: label for label, i in label_to_int.items()}

    y = df["domain_label"].map(label_to_int).values
    seqs = df["sequence"].tolist()
    ids = df["id"].tolist()

    print("\nEncoded classes:")
    for label in LABEL_ORDER:
        print(label_to_int[label], "->", label)

    dataset_name = os.path.splitext(os.path.basename(METADATA_FILE))[0]

    OUTPUT_RESULTS_XLSX = os.path.join(
        OUTPUT_DIR, f"{dataset_name}_sequence_identity_baseline_5fold_results.xlsx"
    )
    OUTPUT_RESULTS_TSV = os.path.join(
        OUTPUT_DIR, f"{dataset_name}_sequence_identity_baseline_5fold_results.tsv"
    )
    OUTPUT_PRED_XLSX = os.path.join(
        OUTPUT_DIR, f"{dataset_name}_sequence_identity_baseline_fold_predictions.xlsx"
    )
    OUTPUT_PER_CLASS_XLSX = os.path.join(
        OUTPUT_DIR, f"{dataset_name}_sequence_identity_baseline_per_class_results.xlsx"
    )
    OUTPUT_LABEL_MAP_XLSX = os.path.join(
        OUTPUT_DIR, f"{dataset_name}_sequence_identity_baseline_label_mapping.xlsx"
    )

    # Cross-validation
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    accs, f1s, precs, recs, aucs = [], [], [], [], []
    all_fold_predictions = []
    per_class_results = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(seqs, y), start=1):
        print("\n" + "=" * 60)
        print(f"Fold {fold}")
        print("=" * 60)

        train_seqs = [seqs[i] for i in train_idx]
        train_labels = [y[i] for i in train_idx]
        train_ids = [ids[i] for i in train_idx]

        test_seqs = [seqs[i] for i in test_idx]
        test_labels = [y[i] for i in test_idx]
        test_ids = [ids[i] for i in test_idx]

        y_pred = []
        y_score = []
        best_identities = []
        best_train_ids = []

        for j, test_seq in enumerate(test_seqs):
            best_identity = -1.0
            best_label = None
            best_train_id = None

            for tr_seq, tr_label, tr_id in zip(train_seqs, train_labels, train_ids):
                ident = calc_sequence_identity(test_seq, tr_seq)

                if ident > best_identity:
                    best_identity = ident
                    best_label = tr_label
                    best_train_id = tr_id

            y_pred.append(best_label)
            best_identities.append(best_identity)
            best_train_ids.append(best_train_id)

            # Use nearest-neighbor identity as score for CAL class if CAL=1
            if int_to_label[best_label] == "CAL":
                y_score.append(best_identity)
            else:
                y_score.append(1.0 - best_identity)

            if (j + 1) % 20 == 0 or (j + 1) == len(test_seqs):
                print(f"  Processed {j+1}/{len(test_seqs)} test sequences")

        acc = accuracy_score(test_labels, y_pred)
        f1 = f1_score(test_labels, y_pred, average="binary", zero_division=0)
        prec = precision_score(test_labels, y_pred, average="binary", zero_division=0)
        rec = recall_score(test_labels, y_pred, average="binary", zero_division=0)

        try:
            auc = roc_auc_score(test_labels, y_score)
        except:
            auc = np.nan

        accs.append(acc)
        f1s.append(f1)
        precs.append(prec)
        recs.append(rec)
        aucs.append(auc)

        print(f"\nFold {fold} metrics:")
        print(f"  Accuracy  = {acc:.4f}")
        print(f"  F1        = {f1:.4f}")
        print(f"  Precision = {prec:.4f}")
        print(f"  Recall    = {rec:.4f}")
        print(f"  ROC-AUC   = {auc:.4f}" if not np.isnan(auc) else "  ROC-AUC   = NA")

        cm = confusion_matrix(test_labels, y_pred, labels=[0, 1])
        cm_df = pd.DataFrame(
            cm,
            index=[f"true_{int_to_label[i]}" for i in [0, 1]],
            columns=[f"pred_{int_to_label[i]}" for i in [0, 1]]
        )
        print("\nConfusion matrix:")
        print(cm_df)

        report = classification_report(
            test_labels,
            y_pred,
            labels=[0, 1],
            target_names=[int_to_label[i] for i in [0, 1]],
            output_dict=True,
            zero_division=0
        )

        print("\nPer-class metrics:")
        for cls_name in LABEL_ORDER:
            print(
                f"  {cls_name}: "
                f"Precision={report[cls_name]['precision']:.4f}, "
                f"Recall={report[cls_name]['recall']:.4f}, "
                f"F1={report[cls_name]['f1-score']:.4f}, "
                f"Support={report[cls_name]['support']:.0f}"
            )

            per_class_results.append({
                "Dataset": dataset_name,
                "fold": fold,
                "Class": cls_name,
                "Precision": report[cls_name]["precision"],
                "Recall": report[cls_name]["recall"],
                "F1_score": report[cls_name]["f1-score"],
                "Support": report[cls_name]["support"]
            })

        fold_pred_df = pd.DataFrame({
            "Dataset": dataset_name,
            "fold": fold,
            "test_id": test_ids,
            "true_label": [int_to_label[v] for v in test_labels],
            "pred_label": [int_to_label[v] for v in y_pred],
            "best_train_id": best_train_ids,
            "best_identity": best_identities
        })
        all_fold_predictions.append(fold_pred_df)

    # Save results
    results_df = pd.DataFrame([{
        "Dataset": dataset_name,
        "Model": "SequenceIdentityNearestNeighbor",
        "Accuracy_mean": np.mean(accs),
        "Accuracy_std": np.std(accs),
        "F1_mean": np.mean(f1s),
        "F1_std": np.std(f1s),
        "Precision_mean": np.mean(precs),
        "Precision_std": np.std(precs),
        "Recall_mean": np.mean(recs),
        "Recall_std": np.std(recs),
        "ROC_AUC_mean": np.nanmean(aucs),
        "ROC_AUC_std": np.nanstd(aucs),
    }])

    per_class_df = pd.DataFrame(per_class_results)
    per_class_summary_df = (
        per_class_df.groupby("Class", as_index=False)
        .agg({
            "Precision": ["mean", "std"],
            "Recall": ["mean", "std"],
            "F1_score": ["mean", "std"],
            "Support": "sum"
        })
    )

    per_class_summary_df.columns = [
        "Class",
        "Precision_mean", "Precision_std",
        "Recall_mean", "Recall_std",
        "F1_score_mean", "F1_score_std",
        "Support_total"
    ]
    per_class_summary_df.insert(0, "Dataset", dataset_name)

    print("\n" + "=" * 60)
    print("Final results")
    print("=" * 60)
    print(results_df)

    print("\n" + "=" * 60)
    print("Per-class summary")
    print("=" * 60)
    print(per_class_summary_df)

    results_df.to_excel(OUTPUT_RESULTS_XLSX, index=False)
    results_df.to_csv(OUTPUT_RESULTS_TSV, sep="\t", index=False)

    pred_df = pd.concat(all_fold_predictions, ignore_index=True)
    pred_df.to_excel(OUTPUT_PRED_XLSX, index=False)

    per_class_summary_df.to_excel(OUTPUT_PER_CLASS_XLSX, index=False)

    label_map_df = pd.DataFrame({
        "encoded_label": [0, 1],
        "domain_class": LABEL_ORDER
    })
    label_map_df.to_excel(OUTPUT_LABEL_MAP_XLSX, index=False)

    print("\nSaved:")
    print(" -", OUTPUT_RESULTS_XLSX)
    print(" -", OUTPUT_RESULTS_TSV)
    print(" -", OUTPUT_PRED_XLSX)
    print(" -", OUTPUT_PER_CLASS_XLSX)
    print(" -", OUTPUT_LABEL_MAP_XLSX)

    all_results.append(results_df)
    all_per_class_summary.append(per_class_summary_df)


# =========================================================
# 5. SAVE COMBINED RESULTS
# =========================================================
if len(all_results) > 0:
    combined_results_df = pd.concat(all_results, ignore_index=True)
    combined_results_df.to_excel(
        os.path.join(OUTPUT_DIR, "combined_sequence_identity_baseline_5fold_results.xlsx"),
        index=False
    )
    combined_results_df.to_csv(
        os.path.join(OUTPUT_DIR, "combined_sequence_identity_baseline_5fold_results.tsv"),
        sep="\t",
        index=False
    )

if len(all_per_class_summary) > 0:
    combined_per_class_df = pd.concat(all_per_class_summary, ignore_index=True)
    combined_per_class_df.to_excel(
        os.path.join(OUTPUT_DIR, "combined_sequence_identity_baseline_per_class_results.xlsx"),
        index=False
    )

print("\nAll done.")


################################################################################
Processing: testset_A_Ak_CAL_table_95.xlsx
################################################################################
Loaded metadata file: testset_A_Ak_CAL_table_95.xlsx
Rows: 518
Columns: ['id', 'sequence', 'domain_label']

Binary label counts:
domain_label
CAL    215
Ak     119
Name: count, dtype: int64

Removed exact duplicate sequence+label rows: 0
Rows after deduplication: 334

Encoded classes:
0 -> Ak
1 -> CAL

Fold 1
  Processed 20/67 test sequences
  Processed 40/67 test sequences
  Processed 60/67 test sequences
  Processed 67/67 test sequences

Fold 1 metrics:
  Accuracy  = 0.7313
  F1        = 0.7857
  Precision = 0.8049
  Recall    = 0.7674
  ROC-AUC   = 0.7074

Confusion matrix:
          pred_Ak  pred_CAL
true_Ak        16         8
true_CAL       10        33

Per-class metrics:
  Ak: Precision=0.6154, Recall=0.6667, F1=0.6400, Support=24
  CAL: Precision=0.8049, Recall=0.7674, F1=0.